In [27]:
#Import libraries
import json
import os
import sys
import time
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pysam
import pyranges as pr

#Define paths
data = "/Users/zhaliedingding/data"

# Load in tools & process

In [ ]:
# #Get hg38 genome fasta file 
# !wget -c https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/latest/hg38.fa.gz
# !gunzip hg38.fa.gz 
# # wget is the terminal downloading command
# # -c is 'continue', which allows the download to continue from last breaking point

In [21]:
# extract 5UTR location from bed file
gtf = pd.read_csv(data + "/2_SCN1A_transcripts_exons_UTR.gtf", sep='\t', header=None)

In [33]:
pd.set_option("display.max_rows", 10)       # 最多显示 100 行
# pd.set_option("display.max_columns", None)   # 显示所有列
# pd.set_option("display.max_colwidth", None)  # 完整显示单元格文字，不用 ...
# display (gtf)

gtf

,0,1,2,3,4,5,6,7,8
0,2,havana,exon,166149047,166149135,.,-,.,"gene_id ""ENSG00000144285""; gene_version ""24""; transcript_id ""ENST00000641575""; transcript_version ""1""; exon_number ""1""; gene_name ""SCN1A""; gene_source ""ensembl_havana""; gene_biotype ""protein_coding""; transcript_name ""SCN1A-218""; transcript_source ""havana""; transcript_biotype ""protein_coding""; tag ""CCDS""; ccds_id ""CCDS86891""; exon_id ""ENSE00003813819""; exon_version ""1""; tag ""gencode_basic""; tag ""gencode_primary"";"
1,2,havana,exon,166077710,166077847,.,-,.,"gene_id ""ENSG00000144285""; gene_version ""24""; transcript_id ""ENST00000641575""; transcript_version ""1""; exon_number ""2""; gene_name ""SCN1A""; gene_source ""ensembl_havana""; gene_biotype ""protein_coding""; transcript_name ""SCN1A-218""; transcript_source ""havana""; transcript_biotype ""protein_coding""; tag ""CCDS""; ccds_id ""CCDS86891""; exon_id ""ENSE00003797124""; exon_version ""1""; tag ""gencode_basic""; tag ""gencode_primary"";"
2,2,havana,exon,166073358,166073670,.,-,.,"gene_id ""ENSG00000144285""; gene_version ""24""; transcript_id ""ENST00000641575""; transcript_version ""1""; exon_number ""3""; gene_name ""SCN1A""; gene_source ""ensembl_havana""; gene_biotype ""protein_coding""; transcript_name ""SCN1A-218""; transcript_source ""havana""; transcript_biotype ""protein_coding""; tag ""CCDS""; ccds_id ""CCDS86891""; exon_id ""ENSE00003761695""; exon_version ""1""; tag ""gencode_basic""; tag ""gencode_primary"";"
3,2,havana,exon,166058570,166058688,.,-,.,"gene_id ""ENSG00000144285""; gene_version ""24""; transcript_id ""ENST00000641575""; transcript_version ""1""; exon_number ""4""; gene_name ""SCN1A""; gene_source ""ensembl_havana""; gene_biotype ""protein_coding""; transcript_name ""SCN1A-218""; transcript_source ""havana""; transcript_biotype ""protein_coding""; tag ""CCDS""; ccds_id ""CCDS86891""; exon_id ""ENSE00002470479""; exon_version ""1""; tag ""gencode_basic""; tag ""gencode_primary"";"
4,2,havana,exon,166056411,166056500,.,-,.,"gene_id ""ENSG00000144285""; gene_version ""24""; transcript_id ""ENST00000641575""; transcript_version ""1""; exon_number ""5""; gene_name ""SCN1A""; gene_source ""ensembl_havana""; gene_biotype ""protein_coding""; transcript_name ""SCN1A-218""; transcript_source ""havana""; transcript_biotype ""protein_coding""; tag ""CCDS""; ccds_id ""CCDS86891""; exon_id ""ENSE00001785173""; exon_version ""1""; tag ""gencode_basic""; tag ""gencode_primary"";"
...,...,...,...,...,...,...,...,...,...
52,2,havana,exon,165999723,165999776,.,-,.,"gene_id ""ENSG00000144285""; gene_version ""24""; transcript_id ""ENST00000674923""; transcript_version ""1""; exon_number ""25""; gene_name ""SCN1A""; gene_source ""ensembl_havana""; gene_biotype ""protein_coding""; transcript_name ""SCN1A-224""; transcript_source ""havana""; transcript_biotype ""protein_coding""; tag ""CCDS""; ccds_id ""CCDS54413""; exon_id ""ENSE00003654618""; exon_version ""1""; tag ""gencode_basic""; tag ""gencode_primary""; tag ""MANE_Select""; tag ""Ensembl_canonical"";"
53,2,havana,exon,165998038,165998175,.,-,.,"gene_id ""ENSG00000144285""; gene_version ""24""; transcript_id ""ENST00000674923""; transcript_version ""1""; exon_number ""26""; gene_name ""SCN1A""; gene_source ""ensembl_havana""; gene_biotype ""protein_coding""; transcript_name ""SCN1A-224""; transcript_source ""havana""; transcript_biotype ""protein_coding""; tag ""CCDS""; ccds_id ""CCDS54413""; exon_id ""ENSE00003585067""; exon_version ""1""; tag ""gencode_basic""; tag ""gencode_primary""; tag ""MANE_Select""; tag ""Ensembl_canonical"";"
54,2,havana,exon,165996013,165996117,.,-,.,"gene_id ""ENSG00000144285""; gene_version ""24""; transcript_id ""ENST00000674923""; transcript_version ""1""; exon_number ""27""; gene_name ""SCN1A""; gene_source ""ensembl_havana""; gene_biotype ""protein_coding""; transcript_name ""SCN1A-224""; transcript_source ""havana""; transcript_biotype ""protein_coding""; tag ""CCDS""; ccds_id 

In [14]:
gtf.loc[0][4] - gtf.loc[0][3]

np.int64(88)

In [49]:
P1b_5UTR_s = gtf.loc[:,8][0]

'gene_id "ENSG00000144285"; gene_version "24"; transcript_id "ENST00000641575"; transcript_version "1"; exon_number "1"; gene_name "SCN1A"; gene_source "ensembl_havana"; gene_biotype "protein_coding"; transcript_name "SCN1A-218"; transcript_source "havana"; transcript_biotype "protein_coding"; tag "CCDS"; ccds_id "CCDS86891"; exon_id "ENSE00003813819"; exon_version "1"; tag "gencode_basic"; tag "gencode_primary";'

In [53]:
gtf.loc[:,8][28]

'gene_id "ENSG00000144285"; gene_version "24"; transcript_id "ENST00000674923"; transcript_version "1"; exon_number "1"; gene_name "SCN1A"; gene_source "ensembl_havana"; gene_biotype "protein_coding"; transcript_name "SCN1A-224"; transcript_source "havana"; transcript_biotype "protein_coding"; tag "CCDS"; ccds_id "CCDS54413"; exon_id "ENSE00002033907"; exon_version "2"; tag "gencode_basic"; tag "gencode_primary"; tag "MANE_Select"; tag "Ensembl_canonical";'

In [19]:
# generate index
fasta_path = data + "/hg38.fa"
pysam.faidx(fasta_path)

''

In [29]:
# load in genome
fasta_open = pysam.Fastafile(fasta_path)

'...'

In [28]:
# extract interval 
with pysam.Fastafile(fasta_path) as genome:
    sequence = genome.fetch("chr2", 180000,200000)
print(sequence)  # Python string, 100 bases

...


In [ ]:
# STORE -> 5UTR region, mutation were going to do, start and end point of the input to model etc. 

# Identify PAM sites

In [ ]:
# search in pysam for PAM - compare with your bed file
# search for *NGG* along the interval (or *CCN*)

# Load in alphagenome tools

# Search strategy